# Policy Remote Function POC (One-Time)

This notebook runs `OPAPolicyEvaluator` against BigQuery traces, with fallback to seeded POC traces.

```mermaid
flowchart LR
  A[Client.evaluate OPAPolicyEvaluator] --> B[BigQuery query on source table]
  B --> C[Remote Function]
  C --> D[Cloud Run OPA service]
  B --> E[policy_decisions_poc]
  E --> F[EvaluationReport + decisions]
```

In [ ]:
import os
from google.cloud import bigquery

PROJECT_ID = os.getenv('GOOGLE_CLOUD_PROJECT', 'rag-chatbot-485501')
DATASET_ID = os.getenv('BQ_DATASET', 'agent_trace')
PRIMARY_TABLE = os.getenv('PRIMARY_TABLE', 'agent_events_v2')
FALLBACK_TABLE = os.getenv('FALLBACK_TABLE', 'agent_events_policy_poc')
LOCATION = os.getenv('BQ_LOCATION', 'US')
REMOTE_FUNCTION_FQN = os.getenv('REMOTE_FUNCTION_FQN', f'{PROJECT_ID}.{DATASET_ID}.opa_policy_eval')

MAX_BYTES_BILLED_GB = int(os.getenv('MAX_BYTES_BILLED_GB', '50'))
MAX_STEP_BUDGET_USD = float(os.getenv('MAX_STEP_BUDGET_USD', '5'))
ESTIMATED_QUERY_USD = (MAX_BYTES_BILLED_GB / 1024.0) * 5.0
if ESTIMATED_QUERY_USD > MAX_STEP_BUDGET_USD:
    raise RuntimeError(
        f'Estimated step cost ${ESTIMATED_QUERY_USD:.2f} exceeds cap ${MAX_STEP_BUDGET_USD:.2f}'
    )

print('Project:', PROJECT_ID)
print('Dataset:', DATASET_ID)
print('Primary table:', PRIMARY_TABLE)
print('Fallback table:', FALLBACK_TABLE)
print('Remote function:', REMOTE_FUNCTION_FQN)
print('Estimated max query cost USD:', round(ESTIMATED_QUERY_USD, 4))

In [ ]:
client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
count_query = f'''
SELECT COUNT(1) AS c
FROM `{PROJECT_ID}.{DATASET_ID}.{PRIMARY_TABLE}`
WHERE timestamp > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOUR)
'''
cfg = bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES_BILLED_GB * 1024**3)
rows = list(client.query(count_query, job_config=cfg).result())
primary_count = int(rows[0]['c']) if rows else 0
active_table = PRIMARY_TABLE if primary_count > 0 else FALLBACK_TABLE
print('Primary count:', primary_count)
print('Active table:', active_table)
if primary_count == 0:
    print('Tip: seed fallback table with:')
    print('python examples/policy_poc/seed_mock_traces_to_bq.py --project', PROJECT_ID)

In [ ]:
from bigquery_agent_analytics import Client, OPAPolicyEvaluator

sdk = Client(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    table_id=active_table,
    location=LOCATION,
    verify_schema=False,
)

evaluator = OPAPolicyEvaluator(
    policy_id='pii_egress_v1',
    policy_version='v1',
    mode='remote_function',
    remote_function_fqn=REMOTE_FUNCTION_FQN,
    persist_table='policy_decisions_poc',
    return_decisions=True,
    lookback_hours=24,
    max_events=2000,
    allow_fallback_seed_table=True,
    fallback_table_id=FALLBACK_TABLE,
    max_bytes_billed_gb=MAX_BYTES_BILLED_GB,
)

report = sdk.evaluate(evaluator=evaluator, dataset=active_table)
print(report.summary())
print('details keys:', sorted(report.details.keys()))
print('persisted table:', report.details.get('persisted_table'))
print('decisions returned:', len(report.details.get('policy_decisions', [])))

In [ ]:
preview = OPAPolicyEvaluator(
    policy_id='pii_egress_v1',
    mode='python_udf_preview',
    enable_preview_python_udf=True,
    persist_table=None,
    return_decisions=False,
    max_bytes_billed_gb=MAX_BYTES_BILLED_GB,
)
print('Preview path is experimental and not parity-guaranteed with OPA/Rego.')
# preview_report = sdk.evaluate(evaluator=preview, dataset=active_table)